# Week 2 — Feature Store

**What this does:** automatically reads `reports/model_comparison.csv` to see
which model won, loads that model, runs every image in the dataset through it
to produce a numeric "summary" of each image (an embedding), and saves all of
them into one shared file. This is the feature store: a single, reusable place
that both future training work and Week 3's serving API read from, instead of
each one recomputing things separately and risking drift between them.

**Before running:** make sure `week2_model_comparison.ipynb` has been run at
least once (so `reports/model_comparison.csv` exists), and both model weight
files exist in `models/`.


In [1]:
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

import mlflow

print("PyTorch:", torch.__version__)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.13.0


In [2]:
PROJECT_ROOT = Path("..").resolve()

MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "manifest.csv"
MODELS_DIR = PROJECT_ROOT / "models"
COMPARISON_PATH = PROJECT_ROOT / "reports" / "model_comparison.csv"
FEATURE_STORE_DIR = PROJECT_ROOT / "feature_store"
FEATURE_STORE_DIR.mkdir(parents=True, exist_ok=True)

CNN_WEIGHTS_PATH = MODELS_DIR / "casting_cnn_best.pth"
RESNET_WEIGHTS_PATH = MODELS_DIR / "resnet18_transfer_best.pth"

for path in [MANIFEST_PATH, COMPARISON_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}\n"
            "Run week2_model_comparison.ipynb first -- this notebook builds on its output."
        )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MLFLOW_DB = PROJECT_ROOT / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB.as_posix()}")
mlflow.set_experiment("casting-defect-model-comparison")

print("Device:", DEVICE)


Device: cpu


## Which model won?

Read straight from the comparison report rather than re-deciding here -- one
source of truth for "the winning model," not two places that could disagree.

In [3]:
comparison_df = pd.read_csv(COMPARISON_PATH)
f1_row = comparison_df[comparison_df["Metric"] == "F1 Score"].iloc[0]

cnn_f1 = f1_row["CNN (from scratch)"]
resnet_f1 = f1_row["ResNet18 (transfer learning)"]

WINNER = "cnn" if cnn_f1 >= resnet_f1 else "resnet"

print(f"CNN F1: {cnn_f1:.4f}")
print(f"ResNet18 F1: {resnet_f1:.4f}")
print(f"Winning model for the feature store: {WINNER.upper()}")


CNN F1: 0.9775
ResNet18 F1: 0.9797
Winning model for the feature store: RESNET


## Load the winning model and its matching preprocessing

Same architecture definitions and preprocessing as the comparison notebook --
copied in directly so this notebook can run completely on its own.

In [4]:
class CastingCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x).squeeze(1)

    def embed(self, x):
        """Returns the 128-dim feature vector, before the final classifier layer."""
        x = self.features(x)
        return torch.flatten(x, 1)


class FeatureDataset(Dataset):
    """Loads every image in the manifest (all splits), with the matching
    preprocessing for whichever model won."""
    def __init__(self, dataframe, mode):
        self.df = dataframe.reset_index(drop=True)
        self.mode = mode  # "cnn" or "resnet"
        self.resnet_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        if self.mode == "cnn":
            image = Image.open(row["processed_filepath"]).convert("L")
            image = image.resize((128, 128), Image.Resampling.BILINEAR)
            image = np.asarray(image, dtype=np.float32) / 255.0
            image = (image - 0.5) / 0.5
            image = torch.from_numpy(np.expand_dims(image, axis=0))
        else:
            image = Image.open(row["processed_filepath"]).convert("RGB")
            image = self.resnet_transform(image)
        return image, index


if WINNER == "cnn":
    model = CastingCNN().to(DEVICE)
    model.load_state_dict(torch.load(CNN_WEIGHTS_PATH, map_location=DEVICE))
    embed_fn = model.embed
    embedding_dim = 128
    model_name = "CastingCNN"
else:
    model = models.resnet18(weights=None)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Linear(num_features, 1), nn.Sigmoid())
    model.load_state_dict(torch.load(RESNET_WEIGHTS_PATH, map_location=DEVICE))
    model.fc = nn.Identity()  # grab the 512-dim features, skip the classifier head
    model.to(DEVICE)
    embed_fn = model
    embedding_dim = 512
    model_name = "ResNet18Transfer"

model.eval()
print(f"Loaded {model_name}, embedding dimension: {embedding_dim}")


Loaded ResNet18Transfer, embedding dimension: 512


## Extract an embedding for every image

Covers all splits (train, val, test) -- a feature store represents the whole
dataset, not just the test set.

In [5]:
manifest = pd.read_csv(MANIFEST_PATH)
dataset = FeatureDataset(manifest, mode=WINNER)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

all_embeddings = np.zeros((len(manifest), embedding_dim), dtype=np.float32)

with torch.no_grad():
    for images, indices in loader:
        images = images.to(DEVICE)
        vectors = embed_fn(images).cpu().numpy()
        all_embeddings[indices.numpy()] = vectors

print(f"Extracted {len(all_embeddings)} embeddings, dimension {embedding_dim}")


Extracted 7284 embeddings, dimension 512


## Build and save the feature store

One row per image: its metadata plus its embedding, versioned with the model
that produced it and a timestamp -- so if the winning model ever changes, old
and new embeddings are never confused with each other.

In [6]:
extracted_at = datetime.now(timezone.utc).isoformat()

feature_df = pd.DataFrame({
    "image_path": manifest["processed_filepath"],
    "split": manifest["split"],
    "label": manifest["label"],
    "label_name": manifest["label_name"],
    "embedding": list(all_embeddings),
    "embedding_dim": embedding_dim,
    "model_name": model_name,
    "extracted_at": extracted_at,
})

feature_store_path = FEATURE_STORE_DIR / "image_embeddings.parquet"
feature_df.to_parquet(feature_store_path, index=False)

print(f"Feature store saved: {feature_store_path}")
print(f"Rows: {len(feature_df)}")
feature_df.drop(columns="embedding").head()


Feature store saved: /Users/dolsynarang/Documents/GitHub/ml-engineering-miniproject/feature_store/image_embeddings.parquet
Rows: 7284


,image_path,split,label,label_name,embedding_dim,model_name,extracted_at
0,../data/processed/test/def_front/cast_def_0_10...,test,1,def_front,512,ResNet18Transfer,2026-08-15T11:51:26.543528+00:00
1,../data/processed/test/def_front/cast_def_0_10...,test,1,def_front,512,ResNet18Transfer,2026-08-15T11:51:26.543528+00:00
2,../data/processed/test/def_front/cast_def_0_10...,test,1,def_front,512,ResNet18Transfer,2026-08-15T11:51:26.543528+00:00
3,../data/processed/test/def_front/cast_def_0_10...,test,1,def_front,512,ResNet18Transfer,2026-08-15T11:51:26.543528+00:00
4,../data/processed/test/def_front/cast_def_0_11...,test,1,def_front,512,ResNet18Transfer,2026-08-15T11:51:26.543528+00:00


## Log it to MLflow

Ties the feature store to the exact model version that produced it --
reproducibility again, this time for the features rather than the model.

In [7]:
with mlflow.start_run(run_name=f"feature-store-{model_name}") as run:
    mlflow.log_params({
        "source_model": model_name,
        "embedding_dim": embedding_dim,
        "num_images": len(feature_df),
    })
    mlflow.log_artifact(str(feature_store_path), artifact_path="feature_store")
    print("Feature store run logged:", run.info.run_id)


Feature store run logged: 1b8a022ac9704506b71454f9f3d6e46b


In [8]:
def get_embedding(image_path: str, store_path: Path = feature_store_path):
    """Look up a precomputed embedding by image path. This is the pattern
    Week 3's serving API uses instead of recomputing features on every request."""
    store = pd.read_parquet(store_path)
    match = store[store["image_path"] == str(image_path)]
    if match.empty:
        return None
    return np.array(match.iloc[0]["embedding"])


sample_path = manifest["processed_filepath"].iloc[0]
sample_embedding = get_embedding(sample_path)
print(f"Looked up embedding for {sample_path}")
print(f"Shape: {sample_embedding.shape}")


Looked up embedding for ../data/processed/test/def_front/cast_def_0_1059.jpeg
Shape: (512,)
